![My MLP](images/mlp.svg)

In [279]:
import numpy as np
import pandas as pd

## Regression

In [280]:
df = pd.DataFrame([[8, 8, 4], [7, 9, 5], [6, 10, 6], [5, 12, 7]], columns=['cgpa', 'profile_score', 'lpa'])
df.head()

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,7


In [281]:
def initialize_parameters(layer_dims):
    np.random.seed(3)

    parameters = {}
    L = len(layer_dims)

    for l in range(1, L):
        parameters['W' + str(l)] = np.ones((layer_dims[l-1], layer_dims[l])) * 0.1
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))

    return parameters

In [282]:
def linear_forward(A_prev, W, b):
    Z = np.dot(W.T, A_prev) + b
    return Z

In [283]:
def sigmoid(Z):
    return (1/(1 + np.exp(-Z)))

In [284]:
def L_layer_forward(X, parameters, linear = True):
    A = X.reshape(-1, 1)
    L = len(parameters) // 2

    for l in range(1, L+1):
        A_prev = A
        W = parameters[f"W{l}"]
        b = parameters[f"b{l}"]
        A = linear_forward(A_prev, W, b)
        
    return A, A_prev

In [285]:
def update_parameters(parameters, y, y_hat, A1, X):
    X = X.reshape(-1, 1)
    lr = 0.001
    step = lr * 2 * (y - y_hat)
    hidden_step = parameters["W2"] @ step

    parameters["W2"] += A1 @ step.T
    parameters["b2"] += step
    parameters["W1"] += X @ hidden_step.T
    parameters["b1"] += hidden_step

In [286]:
X = df[['cgpa', 'profile_score']].to_numpy()
y = df[['lpa']].to_numpy()

parameters = initialize_parameters([2,2,1])

In [287]:
parameters

{'W1': array([[0.1, 0.1],
        [0.1, 0.1]]),
 'b1': array([[0.],
        [0.]]),
 'W2': array([[0.1],
        [0.1]]),
 'b2': array([[0.]])}

In [288]:

for i in range(20):

    loss = []

    for j in range(X.shape[0]):
        k = np.random.randint(X.shape[0])
        y_hat, A1 = L_layer_forward(X[k], parameters)
        update_parameters(parameters, y[k], y_hat, A1, X[k])
        loss.append((y - y_hat) ** 2)

    print(f"Epoch {i+1}", "Loss - ", np.array(loss).mean())




Epoch 1 Loss -  26.486260478568557
Epoch 2 Loss -  21.230739548681214
Epoch 3 Loss -  13.15917867887254
Epoch 4 Loss -  5.097913846026872
Epoch 5 Loss -  1.8054470777310268
Epoch 6 Loss -  1.7704663339017626
Epoch 7 Loss -  1.3547164875356594
Epoch 8 Loss -  1.4230402477672224
Epoch 9 Loss -  1.3263892333998293
Epoch 10 Loss -  1.2779838122166765
Epoch 11 Loss -  1.302443934990946
Epoch 12 Loss -  1.4550335951508027
Epoch 13 Loss -  1.5933411208960615
Epoch 14 Loss -  1.3191708360989909
Epoch 15 Loss -  1.3149430253636694
Epoch 16 Loss -  1.3288394410963846
Epoch 17 Loss -  1.513786255400482
Epoch 18 Loss -  1.4585477013427248
Epoch 19 Loss -  1.3232701944023293
Epoch 20 Loss -  1.3520107801984316


In [289]:
parameters

{'W1': array([[0.20961289, 0.20961289],
        [0.42661352, 0.42661352]]),
 'b1': array([[0.0259196],
        [0.0259196]]),
 'W2': array([[0.46636292],
        [0.46636292]]),
 'b2': array([[0.11754526]])}

In [290]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Input

In [291]:
model = Sequential([
    Input(shape=(2,)),
    Dense(2, activation='linear'),
    Dense(1, activation='linear')
])
optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='mean_squared_error', optimizer=optimizer)

In [292]:
model.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                │ (None, 2)              │             6 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9 (36.00 B)

 Trainable params: 9 (36.00 B)

 Non-trainable params: 0 (0.00 B)

In [293]:
model.get_weights()

[array([[-0.5491902, -0.6972437],
        [ 0.2825998, -0.8475561]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[-1.1714214 ],
        [-0.92258877]], dtype=float32),
 array([0.], dtype=float32)]

In [294]:
weights = [np.array([[0.1, 0.1 ],
                        [0.1, 0.1]], dtype=np.float32),
                np.array([0.0, 0.0], dtype=np.float32),
                np.array([[0.1],
                        [0.1 ]], dtype=np.float32),
                np.array([0], dtype=np.float32)]

In [295]:
model.set_weights(weights)

In [296]:
model.fit(df.iloc[:, :-1].values, df['lpa'].values, epochs=100, batch_size=1)

Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 27.8858  
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 27.5892
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 27.2425
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 26.9156
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 26.5438 
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 26.2091
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 25.8553
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 25.4402
Epoch 9/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 25.0595 
Epoch 10/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 24.6778
Epoch 11/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 24.2610
Epoch 12/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 23.8404
Epoch 13/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 23.3867
Epoch 14/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 22.9342
Epoch 15/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 22.4621 
Epoch 16/100
4

## Classification

In [297]:
df = pd.DataFrame([[8, 8, 1], [7, 9, 1], [6, 10, 0], [5, 5, 0]], columns=['cgpa', 'profile_score', 'placement'])
df.head()

,cgpa,profile_score,placement
0,8,8,1
1,7,9,1
2,6,10,0
3,5,5,0


In [298]:
X = df[['cgpa', 'profile_score']].to_numpy()
y = df['placement'].to_numpy()

parameters = initialize_parameters([2,2,1])

In [299]:
parameters

{'W1': array([[0.1, 0.1],
        [0.1, 0.1]]),
 'b1': array([[0.],
        [0.]]),
 'W2': array([[0.1],
        [0.1]]),
 'b2': array([[0.]])}

In [300]:
def linear_forward(A_prev, W, b):
    Z = np.dot(W.T, A_prev) + b
    A = sigmoid(Z)
    return A

In [301]:
def L_layer_forward(X, parameters, linear = True):
    A = X.reshape(-1, 1)
    L = len(parameters) // 2

    for l in range(1, L+1):
        A_prev = A
        W = parameters[f"W{l}"]
        b = parameters[f"b{l}"]
        A = linear_forward(A_prev, W, b)
        
    return A, A_prev

In [302]:
def update_parameters(parameters, y, y_hat, A1, X):
    X = X.reshape(-1, 1)
    lr = 0.001
    step = lr * (y - y_hat)
    hidden_step = parameters["W2"] @ step * A1 * (1 - A1)

    parameters["W2"] += A1 @ step.T
    parameters["b2"] += step
    parameters["W1"] += X @ hidden_step.T
    parameters["b1"] += hidden_step

In [303]:
for i in range(20):

    loss = []

    for k in np.random.permutation(X.shape[0]):
        y_hat, A1 = L_layer_forward(X[k], parameters)

        update_parameters(parameters, y[k], y_hat, A1, X[k])

        loss.append(-y[k]*np.log(y_hat) - (1-y[k])*np.log(1 - y_hat))

    print(f"Epoch {i + 1} | Loss: {np.array(loss).mean()}")

Epoch 1 | Loss: 0.6941719771434792
Epoch 2 | Loss: 0.694162100838819
Epoch 3 | Loss: 0.694152069908584
Epoch 4 | Loss: 0.694142385087879
Epoch 5 | Loss: 0.6941327415483438
Epoch 6 | Loss: 0.6941231075435845
Epoch 7 | Loss: 0.6941134995196729
Epoch 8 | Loss: 0.6941037731586045
Epoch 9 | Loss: 0.6940943175034717
Epoch 10 | Loss: 0.694084608720188
Epoch 11 | Loss: 0.6940752342068292
Epoch 12 | Loss: 0.6940656299813941
Epoch 13 | Loss: 0.6940563340812687
Epoch 14 | Loss: 0.6940470272615875
Epoch 15 | Loss: 0.694037585669194
Epoch 16 | Loss: 0.6940282350708876
Epoch 17 | Loss: 0.6940192753301782
Epoch 18 | Loss: 0.6940100953618376
Epoch 19 | Loss: 0.6940009922804408
Epoch 20 | Loss: 0.6939919118897343


In [304]:
model = Sequential([
    Input(shape=(2,)),
    Dense(2, activation='sigmoid'),
    Dense(1, activation='sigmoid')
])
optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='binary_crossentropy', optimizer=optimizer)

In [305]:
model.get_weights()

[array([[ 0.69536555,  0.38322246],
        [ 0.22275662, -0.02880538]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[-0.5099391],
        [-0.6125576]], dtype=float32),
 array([0.], dtype=float32)]

In [306]:
weights = [np.array([[0.1, 0.1 ],
                        [0.1, 0.1]], dtype=np.float32),
                np.array([0.0, 0.0], dtype=np.float32),
                np.array([[0.1],
                        [0.1 ]], dtype=np.float32),
                np.array([0], dtype=np.float32)]

In [307]:
model.set_weights(weights)

In [308]:
model.fit(df.iloc[:, :-1].values, df['placement'].values, epochs=100, batch_size=1)

Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6950  
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6938
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6939
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6937
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6937 
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6937
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6937
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6938
Epoch 9/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6937
Epoch 10/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6936
Epoch 11/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6936
Epoch 12/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6935
Epoch 13/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6935
Epoch 14/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6935
Epoch 15/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6934
Epoch 16/100
4/4 ━━━━━━━━━━━━━━